In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

df = pd.read_csv("course_final.csv", encoding="utf-8", index_col=0)

df.head()

,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,gender,grade,...,start_year,age_raw,age_missing_original,age_invalid,age_cleaned,age_final,age,age_group,exam_flag,LoE_num
course_id,,,,,,,,,,,,,,,,,,,,,
HarvardX/PH207x/2012_Fall,MHxPC130275857,1,1,1,0,United States,unknown,NaN,unknown,0,...,2012,NaN,True,False,NaN,NaN,NaN,unknown,1,NaN
HarvardX/CB22x/2013_Spring,MHxPC130539455,1,1,0,0,France,unknown,NaN,unknown,0,...,2013,NaN,True,False,NaN,NaN,NaN,unknown,1,NaN
HarvardX/CB22x/2013_Spring,MHxPC130088379,1,1,0,0,United States,unknown,NaN,unknown,0,...,2013,NaN,True,False,NaN,NaN,NaN,unknown,1,NaN
HarvardX/ER22x/2013_Spring,MHxPC130088379,1,1,0,0,United States,unknown,NaN,unknown,0,...,2013,NaN,True,False,NaN,NaN,NaN,unknown,1,NaN
HarvardX/ER22x/2013_Spring,MHxPC130198098,1,1,0,0,United States,unknown,NaN,unknown,0,...,2013,NaN,True,False,NaN,NaN,NaN,unknown,1,NaN


In [2]:
# sufficient 정의

# 1. 정상 학습 기반 집단 정의
certified_clean = df[
    (df["certified"] == 1) &
    (df["explored"] == 1)
]

# 2. 강의별 nchapters 75% 지점 계산
p75_by_course = (
    certified_clean
    .groupby("course_id")["nchapters"]
    .quantile(0.75)
    .reset_index()
    .rename(columns={"nchapters": "nchapters_p75"})
)

# 3. merge
df = df.merge(p75_by_course, on="course_id", how="left")

# 4. sufficient 생성
df["sufficient"] = (
    (df["nchapters"] >= df["nchapters_p75"]) &
    (df["explored"] == 1)
).astype(int)

In [3]:
conditions = [
    (df['sufficient'] == 1),
    (df['explored'] == 1),
    (df['viewed'] == 1)
]

choices = ['sufficient', 'explored', 'viewed']

df['stage'] = np.select(conditions, choices, default = 'registered')

In [4]:
df['stage'].value_counts()

stage
viewed        287777
registered    208632
explored       25714
sufficient     11870
Name: count, dtype: int64

### 1. viewed -> explored 임계값 계산

In [5]:
df_viewed = df[df["viewed"] == 1].copy()

explored_group = df_viewed[df_viewed["explored"] == 1]
non_explored_group = df_viewed[df_viewed["explored"] == 0]

print("Explored:", len(explored_group))
print("Not Explored:", len(non_explored_group))

Explored: 37584
Not Explored: 287777


In [6]:
def find_thresholds(y_true, scores):
    
    # ROC
    fpr, tpr, roc_thresholds = roc_curve(y_true, scores)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    roc_best_threshold = roc_thresholds[best_idx]
    
    # PR
    precision, recall, pr_thresholds = precision_recall_curve(y_true, scores)
    f1 = (2 * precision * recall) / (precision + recall + 1e-9)
    best_idx_pr = np.argmax(f1)
    
    pr_best_threshold = pr_thresholds[best_idx_pr]
    
    return roc_best_threshold, pr_best_threshold

In [7]:
def make_threshold_df(df, stage, threshold_vars):
    threshold_results = []
    y = df[stage]

    for var in threshold_vars:
        x = df[var].fillna(0)
        roc_t, pr_t = find_thresholds(y, x)
        threshold_results.append({
            "variable": var,
            "roc_threshold": roc_t,
            "pr_threshold": pr_t
        })

    threshold_df = pd.DataFrame(threshold_results)
    return threshold_df

In [8]:
threshold_vars = ["nevents", "ndays_act", "nplay_video"]
v2e_threshold = make_threshold_df(df_viewed, 'explored', threshold_vars)

In [9]:
e2s_threshold = make_threshold_df(explored_group, 'sufficient', threshold_vars)
e2s_threshold

,variable,roc_threshold,pr_threshold
0,nevents,4971.0,1.0
1,ndays_act,40.0,32.0
2,nplay_video,409.0,0.0


In [10]:
# AI 돌림

# 분석할 변수 리스트
threshold_vars = ["nevents", "ndays_act", "nplay_video"]
course_results = []

# 1. 모든 강의 리스트 순회
for course in df['course_id'].unique():
    df_course = df[df['course_id'] == course]
    
    # --- [A] Viewed -> Explored (v2e) 구간 ---
    df_v = df_course[df_course['viewed'] == 1].copy()
    n_viewed_total = len(df_v)
    n_explored_success = df_v['explored'].sum() # 다음 단계(Explored)로 넘어간 사람 수
    
    # --- [B] Explored -> Sufficient (e2s) 구간 ---
    df_e = df_course[df_course['explored'] == 1].copy()
    n_explored_total = len(df_e)
    n_sufficient_success = df_e['sufficient'].sum() # 다음 단계(Sufficient)로 넘어간 사람 수
    
    # 2. 기본 정보(모수) 저장
    row_data = {
        'course_id': course,
        'v2e_total_users': n_viewed_total,
        'v2e_success_users': n_explored_success,
        'e2s_total_users': n_explored_total,
        'e2s_success_users': n_sufficient_success
    }
    
    # 3. V2E 임계값 계산 (ROC & PR 모두)
    # 조건: 모수가 너무 적거나, 성공/실패 클래스 중 하나가 0명이면 에러 나므로 예외 처리
    if n_viewed_total > 10 and n_explored_success > 0 and (n_viewed_total - n_explored_success) > 0:
        for var in threshold_vars:
            roc_t, pr_t = find_thresholds(df_v['explored'], df_v[var].fillna(0))
            row_data[f'v2e_{var}_roc'] = roc_t
            row_data[f'v2e_{var}_pr'] = pr_t
    else:
        for var in threshold_vars:
            row_data[f'v2e_{var}_roc'] = np.nan
            row_data[f'v2e_{var}_pr'] = np.nan

    # 4. E2S 임계값 계산 (ROC & PR 모두)
    if n_explored_total > 10 and n_sufficient_success > 0 and (n_explored_total - n_sufficient_success) > 0:
        for var in threshold_vars:
            roc_t, pr_t = find_thresholds(df_e['sufficient'], df_e[var].fillna(0))
            row_data[f'e2s_{var}_roc'] = roc_t
            row_data[f'e2s_{var}_pr'] = pr_t
    else:
        for var in threshold_vars:
            row_data[f'e2s_{var}_roc'] = np.nan
            row_data[f'e2s_{var}_pr'] = np.nan

    course_results.append(row_data)

# 5. 최종 데이터프레임 생성 (Merge 안 함!)
df_course_summary = pd.DataFrame(course_results)

# 결과 확인
display(df_course_summary.head())

,course_id,v2e_total_users,v2e_success_users,e2s_total_users,e2s_success_users,v2e_nevents_roc,v2e_nevents_pr,v2e_ndays_act_roc,v2e_ndays_act_pr,v2e_nplay_video_roc,v2e_nplay_video_pr,e2s_nevents_roc,e2s_nevents_pr,e2s_ndays_act_roc,e2s_ndays_act_pr,e2s_nplay_video_roc,e2s_nplay_video_pr
0,HarvardX/PH207x/2012_Fall,24215,4321,4321,776,899.0,1490.0,10.0,15.0,89.0,194.0,4428.0,5747.0,35.0,40.0,712.0,712.0
1,HarvardX/CB22x/2013_Spring,16234,547,547,137,578.0,1200.0,9.0,31.0,inf,0.0,1871.0,2569.0,41.0,31.0,inf,0.0
2,HarvardX/ER22x/2013_Spring,32112,3529,3529,732,326.0,639.0,8.0,13.0,inf,0.0,1223.0,1467.0,27.0,27.0,inf,0.0
3,HarvardX/PH278x/2013_Spring,14993,1188,1188,369,565.0,800.0,7.0,12.0,39.0,114.0,1192.0,1192.0,20.0,17.0,82.0,82.0
4,HarvardX/CS50x/2012,41374,8973,8973,4207,32.0,47.0,4.0,6.0,inf,0.0,89.0,40.0,12.0,2.0,inf,0.0


### 2. 각 임계값 병합

In [13]:
# 1. 딕셔너리로 변환
v2e_dict = v2e_threshold.set_index('variable')['pr_threshold'].to_dict()
e2s_dict = e2s_threshold.set_index('variable')['roc_threshold'].to_dict()

# 2. 모든 임계값 컬럼을 빈칸(NaN)으로 초기화
for var in ['nevents', 'ndays_act', 'nplay_video']:
    df[f'v2e_thresh_{var}'] = np.nan
    df[f'e2s_thresh_{var}'] = np.nan

# ==========================================
# 🌟 핵심: 누적 퍼널 원리 적용!
# ==========================================

# 3. V2E 임계값: Viewed 단계 '이상' 진입한 모든 유저에게 부여 (Viewed, Explored, Certified)
# (이 문을 열어본 모든 사람의 성적표에 V2E 커트라인을 적어줌)
v2e_targets = ['viewed', 'explored', 'sufficient']

for var in ['nevents', 'ndays_act', 'nplay_video']:
    df.loc[df['stage'].isin(v2e_targets), f'v2e_thresh_{var}'] = v2e_dict[var]

# 4. E2S 임계값: Explored 단계 '이상' 진입한 모든 유저에게 부여 (Explored, Certified)
# (이 문까지 도달한 사람들의 성적표에만 E2S 커트라인을 추가로 적어줌)
e2s_targets = ['explored', 'sufficient']

for var in ['nevents', 'ndays_act', 'nplay_video']:
    df.loc[df['stage'].isin(e2s_targets), f'e2s_thresh_{var}'] = e2s_dict[var]

# 5. 눈으로 결과 확인
# Registered는 전부 NaN, Viewed는 v2e만, Explored와 Certified는 둘 다 값이 있어야 정상!
print(df[df['stage'] == 'explored'][['stage', 'v2e_thresh_nevents', 'e2s_thresh_nevents']].head(3))

        stage  v2e_thresh_nevents  e2s_thresh_nevents
0    explored              1274.0              4971.0
81   explored              1274.0              4971.0
103  explored              1274.0              4971.0


In [14]:
df[df['stage'] == 'sufficient']

,course_id,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,gender,...,LoE_num,nchapters_p75,sufficient,stage,v2e_thresh_nevents,e2s_thresh_nevents,v2e_thresh_ndays_act,e2s_thresh_ndays_act,v2e_thresh_nplay_video,e2s_thresh_nplay_video
69,HarvardX/CS50x/2012,MHxPC130166581,1,1,1,0,United States,unknown,NaN,unknown,...,NaN,12.0,1,sufficient,1274.0,4971.0,18.0,40.0,170.0,409.0
92,HarvardX/ER22x/2013_Spring,MHxPC130007191,1,1,1,1,Russian Federation,unknown,NaN,unknown,...,NaN,33.0,1,sufficient,1274.0,4971.0,18.0,40.0,170.0,409.0
106,HarvardX/CS50x/2012,MHxPC130074234,1,1,1,0,Germany,unknown,NaN,unknown,...,NaN,12.0,1,sufficient,1274.0,4971.0,18.0,40.0,170.0,409.0
173,HarvardX/CS50x/2012,MHxPC130465525,1,1,1,1,Other Europe,unknown,NaN,unknown,...,NaN,12.0,1,sufficient,1274.0,4971.0,18.0,40.0,170.0,409.0
191,HarvardX/CS50x/2012,MHxPC130561230,1,1,1,0,Egypt,unknown,NaN,unknown,...,NaN,12.0,1,sufficient,1274.0,4971.0,18.0,40.0,170.0,409.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531031,MITx/6.002x/2013_Spring,MHxPC130471525,1,1,1,0,United States,unknown,NaN,f,...,NaN,19.0,1,sufficient,1274.0,4971.0,18.0,40.0,170.0,409.0
531076,MITx/6.002x/2013_Spring,MHxPC130487918,1,1,1,0,United States,Master's,1987.0,f,...,4.0,19.0,1,sufficient,1274.0,4971.0,18.0,40.0,170.0,409.0
531342,MITx/6.00x/2013_Spring,MHxPC130485573,1,1,1,0,United States,Doctorate,1970.0,m,...,5.0,18.0,1,sufficient,1274.0,4971.0,18.0,40.0,170.0,409.0
532118,MITx/14.73x/2013_Spring,MHxPC130413976,1,1,1,0,Indonesia,Secondary,1994.0,m,...,2.0,12.0,1,sufficient,1274.0,4971.0,18.0,40.0,170.0,409.0


In [ ]:
df_course_summary

,course_id,v2e_total_users,v2e_success_users,e2s_total_users,e2s_success_users,v2e_nevents_roc,v2e_nevents_pr,v2e_ndays_act_roc,v2e_ndays_act_pr,v2e_nplay_video_roc,v2e_nplay_video_pr,e2s_nevents_roc,e2s_nevents_pr,e2s_ndays_act_roc,e2s_ndays_act_pr,e2s_nplay_video_roc,e2s_nplay_video_pr
0,HarvardX/PH207x/2012_Fall,24215,4321,4321,776,899.0,1490.0,10.0,15.0,89.0,194.0,4428.0,5747.0,35.0,40.0,712.0,712.0
1,HarvardX/CB22x/2013_Spring,16234,547,547,137,578.0,1200.0,9.0,31.0,inf,0.0,1871.0,2569.0,41.0,31.0,inf,0.0
2,HarvardX/ER22x/2013_Spring,32112,3529,3529,732,326.0,639.0,8.0,13.0,inf,0.0,1223.0,1467.0,27.0,27.0,inf,0.0
3,HarvardX/PH278x/2013_Spring,14993,1188,1188,369,565.0,800.0,7.0,12.0,39.0,114.0,1192.0,1192.0,20.0,17.0,82.0,82.0
4,HarvardX/CS50x/2012,41374,8973,8973,4207,32.0,47.0,4.0,6.0,inf,0.0,89.0,40.0,12.0,2.0,inf,0.0
5,MITx/6.002x/2012_Fall,24922,3006,3006,777,589.0,1591.0,13.0,21.0,65.0,197.0,4763.0,5519.0,43.0,49.0,580.0,602.0
6,MITx/6.002x/2013_Spring,10635,899,899,198,725.0,1945.0,15.0,28.0,69.0,279.0,4557.0,4557.0,51.0,61.0,491.0,491.0
7,MITx/2.01x/2013_Spring,3865,560,560,181,740.0,1664.0,12.0,19.0,64.0,127.0,2833.0,3663.0,30.0,30.0,245.0,168.0
8,MITx/3.091x/2012_Fall,6999,894,894,259,1016.0,2063.0,13.0,22.0,73.0,223.0,6728.0,6728.0,43.0,49.0,615.0,615.0
9,MITx/6.00x/2012_Fall,41369,4181,4181,1543,1327.0,3070.0,17.0,28.0,111.0,245.0,5403.0,5403.0,58.0,57.0,443.0,347.0


In [17]:
# inf 0으로 대체
df_course_summary.loc[df_course_summary['v2e_nplay_video_roc'] == np.inf, 'v2e_nplay_video_roc'] = 0
df_course_summary.loc[df_course_summary['e2s_nplay_video_roc'] == np.inf, 'e2s_nplay_video_roc'] = 0
df_course_summary.head()

,course_id,v2e_total_users,v2e_success_users,e2s_total_users,e2s_success_users,v2e_nevents_roc,v2e_nevents_pr,v2e_ndays_act_roc,v2e_ndays_act_pr,v2e_nplay_video_roc,v2e_nplay_video_pr,e2s_nevents_roc,e2s_nevents_pr,e2s_ndays_act_roc,e2s_ndays_act_pr,e2s_nplay_video_roc,e2s_nplay_video_pr
0,HarvardX/PH207x/2012_Fall,24215,4321,4321,776,899.0,1490.0,10.0,15.0,89.0,194.0,4428.0,5747.0,35.0,40.0,712.0,712.0
1,HarvardX/CB22x/2013_Spring,16234,547,547,137,578.0,1200.0,9.0,31.0,0.0,0.0,1871.0,2569.0,41.0,31.0,0.0,0.0
2,HarvardX/ER22x/2013_Spring,32112,3529,3529,732,326.0,639.0,8.0,13.0,0.0,0.0,1223.0,1467.0,27.0,27.0,0.0,0.0
3,HarvardX/PH278x/2013_Spring,14993,1188,1188,369,565.0,800.0,7.0,12.0,39.0,114.0,1192.0,1192.0,20.0,17.0,82.0,82.0
4,HarvardX/CS50x/2012,41374,8973,8973,4207,32.0,47.0,4.0,6.0,0.0,0.0,89.0,40.0,12.0,2.0,0.0,0.0


In [ ]:
import numpy as np

def apply_course_threshold(df, summary_df, course_id, v2e_metric='pr', e2s_metric='roc'):
    """
    특정 강의를 선택하여 사용자가 지정한 메트릭(ROC 또는 PR)의 임계값을 
    메인 데이터프레임에 누적 논리로 꽂아넣는 함수 ('course_' 이름표 추가)
    """
    # 1. 요약표에서 해당 강의의 데이터 한 줄(Row)
    course_data = summary_df[summary_df['course_id'] == course_id]
    
    if course_data.empty:
        print(f"🚨 [{course_id}] 강의 데이터를 요약표에서 찾을 수 없습니다.")
        return df
        
    course_data = course_data.iloc[0] # 다루기 쉽게 Series 형태로 변환
    
    # 2. 메인 df에 임계값을 담을 그릇(컬럼)이 없다면 미리 빈칸으로 생성
    vars_to_apply = ["nevents", "ndays_act", "nplay_video"]
    for var in vars_to_apply:
        if f'course_v2e_thresh_{var}' not in df.columns:
            df[f'course_v2e_thresh_{var}'] = np.nan
        if f'course_e2s_thresh_{var}' not in df.columns:
            df[f'course_e2s_thresh_{var}'] = np.nan

    # 3. 누적 타겟 그룹 정의
    v2e_targets = ['viewed', 'explored', 'sufficient']
    e2s_targets = ['explored', 'sufficient']
    
    # 4. V2E 임계값 맵핑
    v2e_mask = (df['course_id'] == course_id) & (df['stage'].isin(v2e_targets))
    for var in vars_to_apply:
        target_col_in_summary = f'v2e_{var}_{v2e_metric}' 
        df.loc[v2e_mask, f'course_v2e_thresh_{var}'] = course_data[target_col_in_summary]
        
    # 5. E2S 임계값 맵핑
    e2s_mask = (df['course_id'] == course_id) & (df['stage'].isin(e2s_targets))
    for var in vars_to_apply:
        target_col_in_summary = f'e2s_{var}_{e2s_metric}' 
        df.loc[e2s_mask, f'course_e2s_thresh_{var}'] = course_data[target_col_in_summary]
        
    print(f"✅ [{course_id}] 적용 완료! (V2E: {v2e_metric.upper()} 기준 / E2S: {e2s_metric.upper()} 기준)")
    return df

In [23]:
for course in df.course_id.unique():
    df = apply_course_threshold(df, df_course_summary, course, 'pr', 'roc')

df.head()

✅ [HarvardX/PH207x/2012_Fall] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [HarvardX/CB22x/2013_Spring] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [HarvardX/ER22x/2013_Spring] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [HarvardX/PH278x/2013_Spring] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [HarvardX/CS50x/2012] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [MITx/6.002x/2012_Fall] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [MITx/6.002x/2013_Spring] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [MITx/2.01x/2013_Spring] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [MITx/3.091x/2012_Fall] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [MITx/6.00x/2012_Fall] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [MITx/7.00x/2013_Spring] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [MITx/8.02x/2013_Spring] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [MITx/3.091x/2013_Spring] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [MITx/14.73x/2013_Spring] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [MITx/6.00x/2013_Spring] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)
✅ [MITx/8.MReV/2013_Summer] 적용 완료! (V2E: PR 기준 / E2S: ROC 기준)


,course_id,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,gender,...,v2e_thresh_ndays_act,e2s_thresh_ndays_act,v2e_thresh_nplay_video,e2s_thresh_nplay_video,course_v2e_thresh_nevents,course_e2s_thresh_nevents,course_v2e_thresh_ndays_act,course_e2s_thresh_ndays_act,course_v2e_thresh_nplay_video,course_e2s_thresh_nplay_video
0,HarvardX/PH207x/2012_Fall,MHxPC130275857,1,1,1,0,United States,unknown,NaN,unknown,...,18.0,40.0,170.0,409.0,1490.0,4428.0,15.0,35.0,194.0,712.0
1,HarvardX/CB22x/2013_Spring,MHxPC130539455,1,1,0,0,France,unknown,NaN,unknown,...,18.0,NaN,170.0,NaN,1200.0,NaN,31.0,NaN,0.0,NaN
2,HarvardX/CB22x/2013_Spring,MHxPC130088379,1,1,0,0,United States,unknown,NaN,unknown,...,18.0,NaN,170.0,NaN,1200.0,NaN,31.0,NaN,0.0,NaN
3,HarvardX/ER22x/2013_Spring,MHxPC130088379,1,1,0,0,United States,unknown,NaN,unknown,...,18.0,NaN,170.0,NaN,639.0,NaN,13.0,NaN,0.0,NaN
4,HarvardX/ER22x/2013_Spring,MHxPC130198098,1,1,0,0,United States,unknown,NaN,unknown,...,18.0,NaN,170.0,NaN,639.0,NaN,13.0,NaN,0.0,NaN


In [24]:
df.to_csv('course_tableau.csv', index = False)

In [25]:
df.columns

Index(['course_id', 'userid_DI', 'registered', 'viewed', 'explored',
       'certified', 'final_cc_cname_DI', 'LoE_DI', 'YoB', 'gender', 'grade',
       'start_time_DI', 'last_event_DI', 'nevents', 'ndays_act', 'nplay_video',
       'nchapters', 'nforum_posts', 'viewed_missing_flag', 'duration',
       'fast_completion_flag', 'start_year', 'age_raw', 'age_missing_original',
       'age_invalid', 'age_cleaned', 'age_final', 'age', 'age_group',
       'exam_flag', 'LoE_num', 'nchapters_p75', 'sufficient', 'stage',
       'v2e_thresh_nevents', 'e2s_thresh_nevents', 'v2e_thresh_ndays_act',
       'e2s_thresh_ndays_act', 'v2e_thresh_nplay_video',
       'e2s_thresh_nplay_video', 'course_v2e_thresh_nevents',
       'course_e2s_thresh_nevents', 'course_v2e_thresh_ndays_act',
       'course_e2s_thresh_ndays_act', 'course_v2e_thresh_nplay_video',
       'course_e2s_thresh_nplay_video'],
      dtype='str')